# Detector de Sueño
## Detección de ojos cerrados con notificación

## 1. Instalar dependencias

In [29]:
# Ejecutar solo una vez
!pip install opencv-python tensorflow tf-keras pillow plyer pyttsx3

ERROR: Could not find a version that satisfies the requirement tensorflow (from versions: none)
ERROR: No matching distribution found for tensorflow


## 2. Configuración de rutas

In [30]:
import os

# Ruta relativa al modelo (junto al notebook)
MODELOS_DIR = os.path.join(os.getcwd(), 'modelos_teachablemachine')

MODEL_PATH = os.path.join(MODELOS_DIR, 'converted_keras', 'keras_model.h5')
LABELS_PATH = os.path.join(MODELOS_DIR, 'converted_keras', 'labels.txt')

print('Modelo path:', MODEL_PATH)
print('Model exists:', os.path.exists(MODEL_PATH))

Modelo path: c:\Users\fanny\Documents\UNIBE\Sexto Semestre\Álgebra Lineal\python\modelos_teachablemachine\converted_keras\keras_model.h5
Model exists: True


## 3. Cargar modelos de Teachable Machine

In [31]:
import tf_keras as tf
from PIL import Image
import numpy as np
import shutil
import tempfile

def cargar_modelo(model_path, labels_path):
    """Carga un modelo de Teachable Machine y sus labels."""
    safe_path = model_path
    if any(ord(c) > 127 for c in model_path):
        safe_path = os.path.join(tempfile.gettempdir(), 'tm_model.h5')
        shutil.copy2(model_path, safe_path)
    model = tf.models.load_model(safe_path, compile=False)
    with open(labels_path, 'r') as f:
        labels = [line.strip() for line in f.readlines()]
    print(f'Modelo cargado: {model_path}')
    print(f'Clases: {labels}')
    return model, labels

def predecir(model, img_array, labels):
    """Hace predicción con el modelo de TM. Retorna (clase, confianza)."""
    prediction = model.predict(img_array, verbose=0)
    index = np.argmax(prediction[0])
    confianza = prediction[0][index]
    return labels[index], confianza

def img_to_tm_input(frame, target_size=(224, 224)):
    """Convierte un frame de OpenCV a input para el modelo de TM."""
    img = Image.fromarray(frame).resize(target_size)
    img_array = np.asarray(img, dtype=np.float32)
    if img_array.ndim == 3 and img_array.shape[2] == 3:
        img_array = np.expand_dims(img_array, axis=0)
        img_array = (img_array / 127.5) - 1.0  # Normalización TM [-1, 1]
    return img_array

In [32]:
# Cargar modelo
modelo, labels = cargar_modelo(MODEL_PATH, LABELS_PATH)

Modelo cargado: c:\Users\fanny\Documents\UNIBE\Sexto Semestre\Álgebra Lineal\python\modelos_teachablemachine\converted_keras\keras_model.h5
Clases: ['0 fondo', '1 ojos_abiertos', '2 ojos_cerrados']


## 4. Sistema de Notificaciones

In [33]:
from plyer import notification
import time

ULTIMA_NOTIFICACION = 0
INTERVALO_NOTIFICACION = 10  # segundos entre notificaciones

def enviar_notificacion(mensaje, titulo='🚨 ALERTA DE SUEÑO'):
    """Envía una notificación de escritorio."""
    global ULTIMA_NOTIFICACION
    ahora = time.time()
    if ahora - ULTIMA_NOTIFICACION < INTERVALO_NOTIFICACION:
        return  # No spamear notificaciones
    
    try:
        notification.notify(
            title=titulo,
            message=mensaje,
            app_name='Detector de Sueño',
            timeout=10  # segundos visibles
        )
        ULTIMA_NOTIFICACION = ahora
        print(f'NOTIFICACIÓN: {titulo} - {mensaje}')
    except Exception as e:
        print(f'Error notificación: {e}')

## 5. Detección en Tiempo Real con Webcam

In [34]:
import cv2
import subprocess
import sys
import threading
import time as _time

class LocutorLoop:
    def __init__(self):
        self._activo = threading.Event()
        self._hilo = None

    def _loop(self, mensaje):
        script = (
            'import pyttsx3;'
            'e=pyttsx3.init();'
            'voices=e.getProperty("voices");'
            'spanish=[v for v in voices if "es" in v.id.lower() or "spanish" in v.name.lower()];'
            'e.setProperty("voice", (spanish[0].id if spanish else voices[0].id));'
            'e.setProperty("rate",150);'
            f'e.say("{mensaje}");'
            'e.runAndWait()'
        )
        while self._activo.is_set():
            try:
                p = subprocess.Popen(
                    [sys.executable, '-c', script],
                    creationflags=subprocess.CREATE_NO_WINDOW
                )
                p.wait(timeout=5)
            except Exception:
                pass
            _time.sleep(1)

    def iniciar(self, mensaje):
        if self._hilo and self._hilo.is_alive():
            return
        self._activo.set()
        self._hilo = threading.Thread(target=self._loop, args=(mensaje,), daemon=True)
        self._hilo.start()

    def detener(self):
        self._activo.clear()

locutor = LocutorLoop()

# === CONFIGURACIÓN ===
CONFIAZA_MINIMA = 0.75
FRAMES_OJOS_CERRADOS = 2

contador_ojos_cerrados = 0
alerta_activa = False

print('Iniciando cámara... Presiona Q o cierra la ventana para salir')

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print('ERROR: No se pudo abrir la cámara')
else:
    win_name = 'Detector de Suenio'
    cv2.namedWindow(win_name, cv2.WINDOW_AUTOSIZE)

    while True:
        ret, frame = cap.read()
        if not ret:
            print('No se pudo leer el frame')
            break
        
        frame = cv2.flip(frame, 1)
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        input_img = img_to_tm_input(frame_rgb)
        clase, conf = predecir(modelo, input_img, labels)
        
        estado = 'Normal'
        color = (0, 255, 0)
        
        if 'cerrado' in clase.lower() or 'closed' in clase.lower():
            if conf >= CONFIAZA_MINIMA:
                contador_ojos_cerrados += 1
            else:
                contador_ojos_cerrados = max(0, contador_ojos_cerrados - 1)
        else:
            contador_ojos_cerrados = max(0, contador_ojos_cerrados - 1)
        
        if contador_ojos_cerrados >= FRAMES_OJOS_CERRADOS:
            estado = '¡OJOS CERRADOS!'
            color = (0, 0, 255)
            if not alerta_activa:
                alerta_activa = True
                enviar_notificacion(
                    '¡Estás cerrando los ojos! No te duermas!',
                    titulo='🚨 OJOS CERRADOS'
                )
                locutor.iniciar('¡No te duermas! ¡Abre los ojos!')
        else:
            if alerta_activa:
                locutor.detener()
            alerta_activa = False
        
        h, w = frame.shape[:2]
        cv2.rectangle(frame, (10, 10), (w - 10, 60), (0, 0, 0), -1)
        cv2.putText(frame, f'Estado: {estado}', (20, 45), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
        cv2.putText(frame, f'{clase} ({conf:.0%})', 
                    (10, h - 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
        cv2.putText(frame, f'Contador: {contador_ojos_cerrados}/{FRAMES_OJOS_CERRADOS}',
                    (10, h - 50), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
        
        cv2.imshow(win_name, frame)
        
        # Salir con Q o con la X de la ventana
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            break
        if cv2.getWindowProperty(win_name, cv2.WND_PROP_VISIBLE) < 1:
            break
    
    locutor.detener()
    cap.release()
    cv2.destroyAllWindows()
    print('Cámara cerrada')

Iniciando cámara... Presiona Q o cierra la ventana para salir
NOTIFICACIÓN: 🚨 OJOS CERRADOS - ¡Estás cerrando los ojos! No te duermas!
NOTIFICACIÓN: 🚨 OJOS CERRADOS - ¡Estás cerrando los ojos! No te duermas!
NOTIFICACIÓN: 🚨 OJOS CERRADOS - ¡Estás cerrando los ojos! No te duermas!
NOTIFICACIÓN: 🚨 OJOS CERRADOS - ¡Estás cerrando los ojos! No te duermas!
NOTIFICACIÓN: 🚨 OJOS CERRADOS - ¡Estás cerrando los ojos! No te duermas!
NOTIFICACIÓN: 🚨 OJOS CERRADOS - ¡Estás cerrando los ojos! No te duermas!
Cámara cerrada


## 6. Estructura del modelo

Tu modelo exportado de Teachable Machine debe tener esta estructura:

```
modelos_teachablemachine/
└── converted_keras/
    ├── keras_model.h5
    └── labels.txt
```

### Clases del modelo:
- `ojos_abierto` - Ojos abiertos (estado normal)
- `ojos_cerrados` - Ojos cerrados (alerta de sueño)
- `fondo` - Fondo sin persona

In [35]:
# Verificar estructura del modelo
import os

print('Estructura del modelo:')
print(f'  {MODEL_PATH}')
print(f'  {LABELS_PATH}')
print()
print('Contenido de labels.txt:')
with open(LABELS_PATH, 'r') as f:
    for line in f:
        print(f'  {line.strip()}')

Estructura del modelo:
  c:\Users\fanny\Documents\UNIBE\Sexto Semestre\Álgebra Lineal\python\modelos_teachablemachine\converted_keras\keras_model.h5
  c:\Users\fanny\Documents\UNIBE\Sexto Semestre\Álgebra Lineal\python\modelos_teachablemachine\converted_keras\labels.txt

Contenido de labels.txt:
  0 fondo
  1 ojos_abiertos
  2 ojos_cerrados
